# Retry Attempts Analysis

Analyse retry attempts from retry-specific audit logs (`retry_audit.jsonl`), tracking patterns by operation, failure types, and recovery metrics.

**Primary Data Source:** Dedicated retry logs (more comprehensive than ingest logs)

**Key Metrics:**
- Total retry attempts and recovery rate
- Transient vs hard failure breakdown
- Failures by operation type (Ollama, ChromaDB, Bitbucket, etc.)
- Exhausted retries (failed after all attempts)
- Retry backoff patterns and latency

**Time Window:** Configure in cell 4 using rolling window or explicit datetime range

## Setup: Import Libraries and Configuration

In [ ]:
# Import Required Libraries
from pathlib import Path
import json
from datetime import datetime, timezone, timedelta
import pandas as pd
import numpy as np
from collections import defaultdict
import sys

# Find workspace root by searching for the 'scripts' directory
# This is more reliable in Jupyter environments than using __file__
current = Path.cwd()
workspace_root = None

# Search up to 3 levels for the scripts directory
for _ in range(3):
    if (current / 'scripts').exists() and (current / 'notebooks').exists():
        workspace_root = current
        break
    current = current.parent

# Fallback to current directory if not found
if workspace_root is None:
    workspace_root = Path.cwd()

# Add to path if not already there
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

print(f"✓ Workspace root: {workspace_root}")
print(f"✓ Python {sys.version.split()[0]}")

In [ ]:
# Configuration: Log Files and Time Window

# Retry-specific log paths (now primary source)
RETRY_LOG_PATHS = [
    workspace_root / 'logs' / 'retry_audit.jsonl',
    workspace_root / 'logs' / 'retry.log',
]

# Legacy ingest audit path (may have limited data)
INGEST_LOG_PATHS = [
    workspace_root / 'logs' / 'ingest_audit.jsonl',
]

# Find retry log
RETRY_LOG_PATH = None
for path in RETRY_LOG_PATHS:
    if path.exists():
        RETRY_LOG_PATH = path
        print(f"✓ Using retry log: {RETRY_LOG_PATH}")
        break

if RETRY_LOG_PATH is None:
    print(f"⚠ Warning: No retry log found. Checked: {[str(p) for p in RETRY_LOG_PATHS]}")

# Find ingest log
INGEST_LOG_PATH = None
for path in INGEST_LOG_PATHS:
    if path.exists():
        INGEST_LOG_PATH = path
        print(f"✓ Using ingest audit log: {INGEST_LOG_PATH}")
        break

if INGEST_LOG_PATH is None:
    print(f"⚠ Ingest log not found at: {INGEST_LOG_PATHS}")

print(f"\nNote: Focus on retry logs for accurate analysis. Ingest logs may be incomplete.")

In [ ]:
# DateTime Filtering Configuration
# Set window for analysis. Adjust these as needed.

# Option 1: Use a rolling window (last N minutes)
USE_ROLLING_WINDOW = True
# ROLLING_MINUTES = 60  # Analyse last 60 minutes
ROLLING_MINUTES = 1440  # Analyse last 24 hours

# Option 2: Use explicit datetime window (ISO 8601 format)
# Set these to None to use rolling window instead
# Example: '2026-02-16T10:00:00Z' or '2026-02-16T10:00:00'
WINDOW_START_ISO = None
WINDOW_END_ISO = None

# Helper function to parse ISO 8601 timestamps
def _parse_iso(ts: str) -> datetime:
    """Parse ISO 8601 timestamp strings."""
    ts = ts.strip()
    if ts.endswith('Z'):
        ts = ts.replace('Z', '+00:00')
    try:
        return datetime.fromisoformat(ts)
    except ValueError:
        try:
            return datetime.strptime(ts, '%Y-%m-%dT%H:%M:%S')
        except ValueError:
            return None

# Determine time window
if WINDOW_START_ISO and WINDOW_END_ISO:
    window_start = _parse_iso(WINDOW_START_ISO)
    window_end = _parse_iso(WINDOW_END_ISO)
    if window_start and window_end:
        print(f"✓ Using explicit time window: {window_start} to {window_end}")
    else:
        print("⚠ Invalid ISO timestamps, falling back to rolling window")
        USE_ROLLING_WINDOW = True
else:
    USE_ROLLING_WINDOW = True

if USE_ROLLING_WINDOW:
    window_end = datetime.now(timezone.utc)
    window_start = window_end - timedelta(minutes=ROLLING_MINUTES)
    print(f"✓ Using rolling window: Last {ROLLING_MINUTES} minutes")
    print(f"  Range: {window_start} to {window_end}")

print(f"\n📝 To use explicit window, edit WINDOW_START_ISO and WINDOW_END_ISO above")

## Load Audit Log Data

Read and parse the JSONL audit log file containing retry events.

## Load Retry Audit Logs

Read and parse the JSONL retry audit log file containing dedicated retry events. This is the primary data source.

In [ ]:
# Load Retry Audit Logs with Time Filtering
retry_logs = []
retry_logs_raw = []

if RETRY_LOG_PATH and RETRY_LOG_PATH.exists():
    print(f"Loading retry logs from: {RETRY_LOG_PATH}")
    try:
        with open(RETRY_LOG_PATH, 'r') as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    entry = json.loads(line)
                    retry_logs_raw.append(entry)
                except json.JSONDecodeError as e:
                    print(f"Warning: Failed to parse line {line_num}: {e}")
                    continue
        
        print(f"✓ Loaded {len(retry_logs_raw)} raw retry log entries")
        
        # Filter by time window
        for entry in retry_logs_raw:
            ts_str = entry.get('timestamp')
            if not ts_str:
                continue
            
            ts = _parse_iso(ts_str)
            if ts and window_start <= ts <= window_end:
                retry_logs.append(entry)
        
        print(f"✓ Filtered to {len(retry_logs)} entries within time window")
        
        if retry_logs:
            print(f"Date range (filtered): {retry_logs[0].get('timestamp', 'N/A')} to {retry_logs[-1].get('timestamp', 'N/A')}")
    except Exception as e:
        print(f"Error loading retry log file: {e}")
else:
    print(f"⚠ Retry log not found: {RETRY_LOG_PATH}")
    print("Proceeding with empty data. Run cell 2 to configure retry log path.")

## Categorise Retry Events

Extract and aggregate retry events by type from the filtered retry audit logs.

In [ ]:
# Categorise Retry Events by Type
retry_event_types = defaultdict(list)

for log_entry in retry_logs:
    # Try both 'event' and 'eventType' fields
    event_type = log_entry.get('event') or log_entry.get('eventType', 'unknown')
    retry_event_types[event_type].append(log_entry)

print(f"Total retry events in window: {len(retry_logs)}")
print(f"\nRetry event breakdown:")
for event_type in sorted(retry_event_types.keys()):
    count = len(retry_event_types[event_type])
    print(f"  {event_type}: {count}")

# Display sample event structure
if retry_logs:
    print(f"\nSample retry event structure:")
    print(json.dumps(retry_logs[0], indent=2, default=str))

## Retry Breakdown by Operation & Error Type

Analyse which operations retry most frequently and what error types they encounter.

In [ ]:
# Extract operation and error metadata from retry logs
retry_ops_errors = defaultdict(lambda: defaultdict(int))
operations_list = set()
error_types_list = set()

for log_entry in retry_logs:
    # Handle both formats: new (top-level) and old (metadata nested)
    metadata = log_entry.get('metadata', {})
    
    # Try top-level first (new format), then metadata (old format)
    operation = log_entry.get('operation') or metadata.get('operation', 'unknown')
    error_type = log_entry.get('error_type') or metadata.get('error_type', 'unknown')
    
    operations_list.add(operation)
    error_types_list.add(error_type)
    retry_ops_errors[operation][error_type] += 1

print("=" * 80)
print("RETRIES BY OPERATION")
print("=" * 80)

# Sort operations by total retry count
op_totals = {op: sum(errors.values()) for op, errors in retry_ops_errors.items()}
for operation in sorted(op_totals.keys(), key=lambda x: op_totals[x], reverse=True):
    total = op_totals[operation]
    print(f"\n{operation}: {total} retries")
    
    # Show error breakdown for this operation
    errors_for_op = retry_ops_errors[operation]
    for error_type in sorted(errors_for_op.keys(), key=lambda x: errors_for_op[x], reverse=True):
        count = errors_for_op[error_type]
        percentage = (count / total * 100) if total > 0 else 0
        print(f"  → {error_type}: {count} ({percentage:.1f}%)")

print("\n" + "=" * 80)
print("RETRIES BY ERROR TYPE")
print("=" * 80)

# Aggregate by error type
error_totals = defaultdict(int)
for op, errors in retry_ops_errors.items():
    for error_type, count in errors.items():
        error_totals[error_type] += count

total_retries = sum(error_totals.values())
for error_type in sorted(error_totals.keys(), key=lambda x: error_totals[x], reverse=True):
    count = error_totals[error_type]
    percentage = (count / total_retries * 100) if total_retries > 0 else 0
    print(f"{error_type}: {count} ({percentage:.1f}%)")

print(f"\nTotal retries across all operations: {total_retries}")
print(f"Unique operations: {len(operations_list)}")
print(f"Unique error types: {len(error_types_list)}")

## Operation × Error Type Matrix

View retry distribution as a cross-tabulation for pattern analysis.

In [ ]:
# Build cross-tabulation matrix
if retry_logs and len(operations_list) > 0 and len(error_types_list) > 0:
    # Create matrix data
    matrix_data = []
    for operation in sorted(operations_list):
        row = {'operation': operation}
        for error_type in sorted(error_types_list):
            row[error_type] = retry_ops_errors[operation].get(error_type, 0)
        row['total'] = sum(row.get(et, 0) for et in error_types_list)
        matrix_data.append(row)
    
    # Sort by total descending
    matrix_data.sort(key=lambda x: x['total'], reverse=True)
    
    # Create DataFrame
    df_matrix = pd.DataFrame(matrix_data)
    
    print("Operation × Error Type Cross-Tabulation:")
    print("(Rows: Operations, Columns: Error Types, Values: Retry Count)\n")
    print(df_matrix.to_string(index=False))
    
    # Identify hotspots (operation-error pairs with high counts)
    print("\n" + "=" * 80)
    print("TOP RETRY HOTSPOTS (Operation × Error Type Pairs)")
    print("=" * 80)
    
    hotspots = []
    for operation in operations_list:
        for error_type in error_types_list:
            count = retry_ops_errors[operation].get(error_type, 0)
            if count > 0:
                hotspots.append((operation, error_type, count))
    
    # Sort by count descending
    hotspots.sort(key=lambda x: x[2], reverse=True)
    
    for i, (op, err, count) in enumerate(hotspots[:15], 1):  # Top 15
        pct = (count / total_retries * 100) if total_retries > 0 else 0
        print(f"{i:2d}. {op:30s} × {err:25s}: {count:4d} ({pct:5.2f}%)")
else:
    print("No retry data available for matrix analysis")

## Note: Legacy Ingest Logs (Optional)

The ingest audit logs (`ingest_audit.jsonl`) previously contained retry events but are now less reliable for retry analysis since dedicated retry logs exist. Use the cells below only if investigating old logs or comparing sources.

In [ ]:
# Load Legacy Ingest Log Data (Optional)
logs = []

print(f"Loading logs from: {INGEST_LOG_PATH}")
if INGEST_LOG_PATH and INGEST_LOG_PATH.exists():
    try:
        with open(INGEST_LOG_PATH, 'r') as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    entry = json.loads(line)
                    logs.append(entry)
                except json.JSONDecodeError as e:
                    print(f"Warning: Failed to parse line {line_num}: {e}")
                    continue
        print(f"✓ Loaded {len(logs)} log entries")
        if logs:
            print(f"Date range: {logs[0].get('timestamp', 'N/A')} to {logs[-1].get('timestamp', 'N/A')}")
    except Exception as e:
        print(f"Error loading log file: {e}")
else:
    print(f"⚠ Log file does not exist: {INGEST_LOG_PATH}")
    print("Note: Subsequent cells will work with empty data")

## Extract Retry Events

Filter and aggregate retry-related audit entries, categorising by event type.

In [ ]:
# Extract Retry Events
retry_events = [
    log for log in logs 
    if log.get('module') == 'retry' and 'event' in log
]

# Categorise by event type
event_types = defaultdict(list)
for event in retry_events:
    event_type = event.get('event')
    event_types[event_type].append(event)

print(f"Total retry events: {len(retry_events)}")
print(f"\nEvent breakdown:")
for event_type in sorted(event_types.keys()):
    count = len(event_types[event_type])
    print(f"  {event_type}: {count}")

# Display sample events
if retry_events:
    print(f"\nSample event structure:")
    print(json.dumps(retry_events[0], indent=2, default=str))

## Analyse Retry Attempt Failures

Extract retry metrics from failed attempts, including operation names, error types, and attempt counts.

In [ ]:
# Analyse Retry Attempt Failures
attempt_failures = event_types.get('retry_attempt_failed', [])
print(f"Retry attempt failures: {len(attempt_failures)}\n")

# Build dataframe for analysis
attempt_data = []
for event in attempt_failures:
    metadata = event.get('metadata', {})
    attempt_data.append({
        'timestamp': event.get('timestamp'),
        'operation': metadata.get('operation'),
        'attempt': metadata.get('attempt'),
        'max_retries': metadata.get('max_retries'),
        'error_type': metadata.get('error_type'),
        'failure_type': metadata.get('failure_type'),
        'will_retry': metadata.get('will_retry'),
    })

if attempt_data:
    df_attempts = pd.DataFrame(attempt_data)
    
    print("Failures by operation:")
    print(df_attempts['operation'].value_counts())
    
    print("\n\nFailures by error type:")
    print(df_attempts['error_type'].value_counts())
    
    print("\n\nFailure type breakdown:")
    print(df_attempts['failure_type'].value_counts())
else:
    print("No retry attempt failures found")

## Retry Success Analysis

Count retry successes—operations that recovered after initial failure through retries.

In [ ]:
# Retry Success Analysis
retry_successes = event_types.get('retry_success', [])
print(f"Successful retries: {len(retry_successes)}\n")

# Build dataframe for successes
success_data = []
for event in retry_successes:
    metadata = event.get('metadata', {})
    success_data.append({
        'timestamp': event.get('timestamp'),
        'operation': metadata.get('operation'),
        'attempt': metadata.get('attempt'),
        'total_attempts': metadata.get('total_attempts'),
    })

if success_data:
    df_success = pd.DataFrame(success_data)
    
    print("Successes by operation:")
    print(df_success['operation'].value_counts())
    
    print("\n\nAttempt distribution (which retry attempt succeeded):")
    print(df_success['attempt'].value_counts().sort_index())
else:
    print("No retry successes found")

## Exhausted Retries & Hard Failures

Track operations that failed despite all retries, and hard failures that never retried.

In [ ]:
# Exhausted Retries & Hard Failures
exhausted_retries = event_types.get('retry_exhausted', [])
hard_failures = event_types.get('hard_failure', [])

print(f"Exhausted retries: {len(exhausted_retries)}")
print(f"Hard failures: {len(hard_failures)}\n")

# Exhausted retries breakdown
if exhausted_retries:
    exhausted_data = []
    for event in exhausted_retries:
        metadata = event.get('metadata', {})
        exhausted_data.append({
            'operation': metadata.get('operation'),
            'max_retries': metadata.get('max_retries'),
            'final_error_type': metadata.get('final_error_type'),
        })
    
    df_exhausted = pd.DataFrame(exhausted_data)
    print("Exhausted retries by operation:")
    print(df_exhausted['operation'].value_counts())
    print("\n")

# Hard failures breakdown
if hard_failures:
    hard_data = []
    for event in hard_failures:
        metadata = event.get('metadata', {})
        hard_data.append({
            'operation': metadata.get('operation'),
            'attempt': metadata.get('attempt'),
            'error_type': metadata.get('error_type'),
        })
    
    df_hard = pd.DataFrame(hard_data)
    print("Hard failures by operation:")
    print(df_hard['operation'].value_counts())
    print("\n")
    print("Hard failures by error type:")
    print(df_hard['error_type'].value_counts())

## Operation Summary

Comprehensive view of retry performance across all operations.

In [ ]:
# Operation Summary
operations = defaultdict(lambda: {'attempts': 0, 'successes': 0, 'exhausted': 0, 'hard_failures': 0})

for event in attempt_failures:
    op = event.get('metadata', {}).get('operation', 'unknown')
    operations[op]['attempts'] += 1

for event in retry_successes:
    op = event.get('metadata', {}).get('operation', 'unknown')
    operations[op]['successes'] += 1

for event in exhausted_retries:
    op = event.get('metadata', {}).get('operation', 'unknown')
    operations[op]['exhausted'] += 1

for event in hard_failures:
    op = event.get('metadata', {}).get('operation', 'unknown')
    operations[op]['hard_failures'] += 1

# Build summary dataframe
summary_data = []
for op, stats in sorted(operations.items()):
    total = stats['attempts'] + stats['exhausted'] + stats['hard_failures']
    success_rate = (stats['successes'] / stats['attempts'] * 100) if stats['attempts'] > 0 else 0
    summary_data.append({
        'operation': op,
        'total_failures': stats['attempts'],
        'succeeded_after_retry': stats['successes'],
        'retry_success_rate_%': round(success_rate, 2),
        'exhausted_retries': stats['exhausted'],
        'hard_failures': stats['hard_failures'],
    })

if summary_data:
    df_summary = pd.DataFrame(summary_data).sort_values('total_failures', ascending=False)
    print("Operation Performance Summary:")
    print(df_summary.to_string(index=False))
else:
    print("No operations recorded—no retry events found in logs.")

## Overall Metrics

Key performance indicators for the entire retry system.

In [ ]:
# Overall Metrics
print("=== OVERALL RETRY METRICS ===\n")

total_attempts = len(attempt_failures)
total_successes = len(retry_successes)
total_exhausted = len(exhausted_retries)
total_hard = len(hard_failures)

print(f"Total retry attempt failures:  {total_attempts}")
print(f"Recovered via retry:            {total_successes}")
print(f"Retries exhausted (still fail):  {total_exhausted}")
print(f"Hard failures (no retry):        {total_hard}")
print()

if total_attempts > 0:
    recovery_rate = (total_successes / total_attempts * 100)
    print(f"Recovery rate (of failed attempts): {recovery_rate:.1f}%")

total_operations = len(operations)
print(f"Unique operations tracked: {total_operations}")

# Identify most problematic operations
if 'df_summary' in locals() and df_summary is not None and len(df_summary) > 0:
    worst_op = df_summary.iloc[0]
    print(f"\nMost problematic operation: {worst_op['operation']}")
    print(f"  Total failures: {worst_op['total_failures']}")
    print(f"  Success rate: {worst_op['retry_success_rate_%']}%")